[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sear-labs/energy-system-modeling/blob/main/notebooks/p0_start/00_start_here.ipynb)

# Start here

## Energy System Modeling - the whole series in one run

Twelve notebooks in this repository teach twelve models. This one teaches nothing. It runs all of them, on fixed data, and prints one number from each.

Open it first. It takes under a minute and it answers the question you actually have on arrival: **does any of this work, and what is in it?**


## Setup

One cell, the same in every notebook here: it installs what Colab does not have, fetches this repository so `data/` and `src/` are present, and moves into this notebook's own folder so the relative paths below resolve. The notebook's own imports follow in the same cell.


In [1]:
# --- setup: generated by tools/sync_setup_cells.py -- do not edit here, edit that
# The same cell in every notebook in this series. It installs what Colab does
# not have, fetches the repository so that data/ and src/ are present, and moves
# into this notebook's own folder so the relative paths below resolve.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sear-labs/energy-system-modeling"
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1]
NOTEBOOK_DIR = "notebooks/p0_start"

# Pinned, per Part 1 rule 3: an unpinned install will one day pull a major
# version with a changed API and either break or silently alter the answer.
PINS = []

if "google.colab" in sys.modules:
    if PINS:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    # An ABSOLUTE base, so re-running this cell is safe. Colab's "Run all" is
    # commonly run twice, and a relative check would look for the clone inside
    # the folder it had already moved into -- cloning a second copy nested one
    # level down, then working from the wrong one.
    BASE = Path("/content") if Path("/content").is_dir() else Path.home()
    REPO_DIR = BASE / REPO_NAME
    if not REPO_DIR.exists():
        cloned = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
            capture_output=True, text=True)
        if cloned.returncode != 0:
            raise SystemExit(
                "Could not clone " + REPO_URL + "\n"
                + (cloned.stderr or "").strip() + "\n\n"
                "If that says 'not found', the repository is still private.\n"
                "A raw file URL fails the same way, so there is no way around\n"
                "it: it has to be public before a student can run this.")
    os.chdir(REPO_DIR / NOTEBOOK_DIR)
elif Path.cwd().name != Path(NOTEBOOK_DIR).name:
    raise SystemExit(
        "Run this notebook from its own folder (" + NOTEBOOK_DIR + "),\n"
        "so that ../../src and ../../data resolve.")

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
print("working directory:", Path.cwd().name)

# --- end generated setup; the notebook's own imports follow ---


working directory: p0_start


### What makes this notebook different from the other twelve

| | the twelve | this one |
|---|---|---|
| purpose | teach a model | show the results |
| method | build it by hand, one cell per step | call the library |
| data | fixed instances, sometimes edited to explore | fixed, untouched |
| endings | several stop at a deliberate blank | runs top to bottom |

Because it stops nowhere and builds nothing by hand, there is no exercise in it to spoil - which is why it is the one notebook that can be shown to anybody.


### The idea the whole series rests on

Every one of the twelve builds a model **by hand**, narrated a step at a time, and then ends by solving the same instance a **second, independent way** and asserting the two agree.

The second way is deliberately a different route, not a second copy: where a notebook writes the linear program in `gurobipy`, the package solves it in `PyPSA`; where a notebook uses `PyPSA`, the package uses `scipy`; where a notebook fits with NumPy's least squares, the package uses SciPy's, through a different LAPACK driver. Two transcriptions of the same algebra share their mistakes. Two genuinely different routes do not.

What runs below is the package half of every one of those pairs.


---
## Setup


---
## Part 1 - Accounting, before anything is optimised

Two of the twelve do arithmetic rather than optimisation. They come first in the course for the same reason they come first here: you cannot optimise a system you cannot add up.


### A household energy balance

Three utility bills in three different units, converted to one, split into the part that did the job and the part that did not. Taught in `p1_foundations/02_one_house_balance`.


In [2]:
from esm.balance import (conservation_residual, load_balance_instance,
                        rejected_share, solve_balance)

house = solve_balance(load_balance_instance())
print(f'energy in    {house.total_in:>9,.0f} kWh')
print(f'useful       {house.total_useful:>9,.1f} kWh  '
      f'{house.total_useful / house.total_in:.0%}')
print(f'rejected     {house.total_rejected:>9,.1f} kWh  '
      f'{house.total_rejected / house.total_in:.0%}')
print(f'residual     {conservation_residual(house)["total"]:>9.1e} kWh')
print()
print('largest single source of waste: '
      f'{list(rejected_share(house))[0]}')


energy in       44,430 kWh
useful        28,155.5 kWh  63%
rejected      16,274.5 kWh  37%
residual       0.0e+00 kWh

largest single source of waste: gasoline


### Screening-level LCOE

What a megawatt-hour costs before anybody optimises anything. Taught in `p3_generation/09_capital_and_lcoe`.


In [3]:
from esm.lcoe import lcoe_by_crf, lcoe_by_dcf, load_lcoe_instance

screen = load_lcoe_instance()
for key in (('coal', 10000), ('gas', 7500)):
    plant = screen.plants[key]
    print(f'{plant.fuel:5s} break-even  '
          f'${lcoe_by_crf(screen, plant):6.2f} /MWh')

gas = screen.plants[('gas', 7500)]
print()
print('the same gas plant, if its output fell half a percent a year:')
print(f'  shortcut says   ${lcoe_by_crf(screen, gas):6.2f} /MWh')
print(f'  definition says '
      f'${lcoe_by_dcf(screen, gas, degradation=0.005):6.2f} /MWh')


coal  break-even  $ 55.48 /MWh
gas   break-even  $ 29.50 /MWh

the same gas plant, if its output fell half a percent a year:
  shortcut says   $ 29.50 /MWh
  definition says $ 30.96 /MWh


---
## Part 2 - Optimisation

> **Predict before you run these.** Three of the four below are least-cost problems on small networks. Write down which you expect to be limited by capacity and which by geography, then see.


### Least-cost transport

Crude oil from two fields to three refineries, over pipelines with capacities. Taught in `p4_networks/17_pipeline_transport`.


In [4]:
from esm.transport import (binding_routes, load_transport_instance,
                          solve_transport)

pipes = load_transport_instance()
ship = solve_transport(pipes)
print(f'least-cost shipping  ${ship.cost:,.0f} /day')
for route in binding_routes(pipes, ship):
    print(f'  at capacity: {route[0]} -> {route[1]}')


least-cost shipping  $490,000 /day
  at capacity: Eagle Ford -> Corpus


### Two-stage sourcing, and what resilience costs

Cobalt from mines through refineries to cell plants, then the same problem with a cap on how much any one mine may supply. Taught in `p5_storage_supply/21_material_requirements`.


In [5]:
from esm.sourcing import load_sourcing_instance, solve_sourcing

chain = load_sourcing_instance()
base = solve_sourcing(chain)
print(f'least cost           ${base.cost:>8,.0f} /yr')
print(f'sourced from         '
      f'{max(base.by_mine, key=base.by_mine.get)}, alone')
print()
print('what it costs to stop depending on one mine:')
for cap in (0.75, 0.60, 0.50, 0.40):
    capped = solve_sourcing(chain, cap)
    print(f'  no more than {cap:.0%} from any one  '
          f'${capped.cost:>8,.0f} /yr   '
          f'+{capped.cost / base.cost - 1:.0%}')


least cost           $     600 /yr
sourced from         DRC, alone

what it costs to stop depending on one mine:
  no more than 75% from any one  $     660 /yr   +10%
  no more than 60% from any one  $     696 /yr   +16%
  no more than 50% from any one  $     720 /yr   +20%
  no more than 40% from any one  $     768 /yr   +28%


### Hourly dispatch, and whether a new load pays its own price

A day of demand met from a merit order, solved with and without a new site attached. Taught in `p1_foundations/01_model_boundary`.


In [6]:
from esm.boundary import bill, load_boundary_instance, solve_dispatch

grid = load_boundary_instance()
alone = solve_dispatch(grid, 0)
print(f'system cost, no new load   ${alone.cost:>12,.0f} /day')
print()
for mw, who in ((50, 'an industrial park'), (500, 'a data centre')):
    withit = solve_dispatch(grid, mw)
    moved = sum(1 for h in grid.hours
                if abs(withit.lmp[h] - alone.lmp[h]) > 1e-9)
    predicted, actual = bill(alone, mw), bill(withit, mw)
    print(f'{who:20s} {mw:>4} MW')
    print(f'   expected to pay  ${predicted:>10,.0f} /day')
    print(f'   actually pays    ${actual:>10,.0f} /day  '
          f'({actual / predicted - 1:+.0%})')
    print(f'   moved the price in {moved} of {len(grid.hours)} hours')


system cost, no new load   $   1,669,378 /day

an industrial park     50 MW
   expected to pay  $    49,500 /day
   actually pays    $    51,300 /day  (+4%)
   moved the price in 3 of 24 hours
a data centre         500 MW
   expected to pay  $   495,000 /day
   actually pays    $   912,000 /day  (+84%)
   moved the price in 19 of 24 hours


### A facility deciding whether to build solar

The same arithmetic a plant manager actually faces, where the answer turns on the tariff rather than on the panels. Taught in `capstone/AppA_facility_decision`.


In [7]:
from esm.facility import annual_bill, load_facility_instance, solve_site

site = load_facility_instance()
do_nothing = annual_bill(site)
with_solar = solve_site(site, solar_max=site.roof_mw)
print(f'bill as things stand   ${do_nothing / 1e6:>7.3f} M/yr')
print(f'with the roof built    ${with_solar.cost / 1e6:>7.3f} M/yr')
print(f'saving                 ${(do_nothing - with_solar.cost) / 1e6:>7.3f} M/yr')
print(f'solar built            {with_solar.solar_mw:>7.1f} MW')
print()
no_charge = solve_site(site, solar_max=site.roof_mw, demand_charge=0.0)
print('and the same site on a tariff with no demand charge:')
print(f'  solar built          {no_charge.solar_mw:>7.1f} MW')


bill as things stand   $ 23.485 M/yr
with the roof built    $ 23.048 M/yr
saving                 $  0.437 M/yr
solar built               12.0 MW

and the same site on a tariff with no demand charge:
  solar built              0.0 MW


---
## Part 3 - Real data, which arrives broken

The models above run on instances small enough to check by eye. Real network data is not, and it does not announce its defects.


### A 2000-bus synthetic Texas grid

Parsed from the MATPOWER case vendored in `data/vendor/`, with the fuel labels that both standard converters throw away. Taught in `p4_networks/15_real_network_import`.


In [8]:
from pathlib import Path

from esm.network_import import (check_matpower_parse, generation_mix,
                               parse_matpower)

case = Path('../../data/vendor/case_ACTIVSg2000.m')
ppc = parse_matpower(case.read_text(encoding='utf-8'))
check_matpower_parse(ppc)
print(f'{len(ppc["bus"]):,} buses, {len(ppc["gen"])} generators, '
      f'{len(ppc["branch"]):,} branches')
print()
print('installed capacity by fuel, MW:')
for fuel, mw in generation_mix(ppc).items():
    print(f'  {fuel:10s} {mw:>9,.0f}')


2,000 buses, 544 generators, 3,206 branches

installed capacity by fuel, MW:
  ng            63,810
  coal          14,502
  wind           9,587
  nuclear        5,139
  hydro          2,603
  solar            651


### Distances, checked against the thing that knows them

Corridor lengths recomputed from hub coordinates by a different formula from the one the notebook uses - the check that catches a transposed latitude. Taught in `capstone/AppA_texas_multi_city_buildout`.


In [9]:
from esm.buildout import (check_demand_shares, coal_cost_per_mwh,
                         corridor_distances, gas_cost_per_mwh,
                         load_buildout_instance)

texas = load_buildout_instance()
check_demand_shares(texas)
print('corridor lengths, km:')
for (a, b), km in corridor_distances(texas).items():
    print(f'  {a[:14]:14s} - {b[:14]:14s} {km:>6.0f}')
print()
print(f'gas  ${gas_cost_per_mwh(texas):5.2f} /MWh-e all in')
print(f'coal ${coal_cost_per_mwh(texas):5.2f} /MWh-e all in')


corridor lengths, km:
  West Texas     - Dallas-Fort Wo    503
  West Texas     - Austin            455
  Dallas-Fort Wo - Houston           362
  Dallas-Fort Wo - Austin            293
  Houston        - Austin            235
  Austin         - San Antonio       118
  Houston        - San Antonio       304

gas  $18.94 /MWh-e all in
coal $19.88 /MWh-e all in


---
## Part 4 - Does it hold together?

Everything above came from the library. In each of the twelve notebooks, the same instance is also built **by hand**, and the notebook asserts the two agree to `AGREEMENT_RTOL`.

This last cell checks that the library still produces what its own tests pin it to. If any line below is not `ok`, something has drifted and the notebook that teaches it will fail too.


In [10]:
from esm.tolerance import AGREEMENT_RTOL, relative

# Full precision, not the rounded figures printed above. A pin typed to
# fewer digits than it has is a pin that cannot be checked tightly -- and
# writing one out by eye is how invented digits get in.
pinned = [
    ('house balance, kWh in', house.total_in, 44430.0),
    ('coal break-even, $/MWh',
     lcoe_by_crf(screen, screen.plants[('coal', 10000)]),
     55.47917398532572),
    ('transport, $/day', ship.cost, 490000.0),
    ('sourcing, $/yr', base.cost, 600.0),
    ('dispatch, $/day', alone.cost, 1669377.7545368262),
    ('facility bill, $/yr', do_nothing, 23484661.71359483),
    ('buses parsed', float(len(ppc['bus'])), 2000.0),
]

worst = max(relative(got, want) for _, got, want in pinned)
for label, got, want in pinned:
    ok = 'ok' if relative(got, want) < AGREEMENT_RTOL else 'DRIFTED'
    print(f'{label:26s} {got:>16,.4f}  {ok}')

assert worst < AGREEMENT_RTOL, (
    f'the library has drifted from its pinned results by {worst:.2e}')
print()
print(f'every model matches its pinned result to {worst:.1e}')


house balance, kWh in           44,430.0000  ok
coal break-even, $/MWh              55.4792  ok
transport, $/day               490,000.0000  ok
sourcing, $/yr                     600.0000  ok
dispatch, $/day              1,669,377.7545  ok
facility bill, $/yr         23,484,661.7136  ok
buses parsed                     2,000.0000  ok

every model matches its pinned result to 0.0e+00


---
### Where to go next

| you want | open |
|---|---|
| the accounting, from nothing | `p1_foundations/02_one_house_balance` |
| what a model boundary is | `p1_foundations/01_model_boundary` |
| your own meter data | `p2_demand/05_representative_days` |
| costs before optimisation | `p3_generation/09_capital_and_lcoe` |
| the transport problem | `p4_networks/17_pipeline_transport` |
| prices on a network | `p4_networks/18_power_flow_and_lmp` |
| a real network, with its defects | `p4_networks/15_real_network_import` |
| a decision to defend | `capstone/AppA_facility_decision` |
| the same model in two tools | `graduate/model_diversity` |

Each of those builds its model by hand first. This notebook called the finished library; they are where the library comes from.
